# Voting and Stacking Classifier

## Import Libraries

In [1]:

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import VotingClassifier, StackingClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


## Load Dataset

In [2]:

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target
X.head()


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


## Train-Test Split

In [3]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## Feature Scaling

In [4]:

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Base Models

In [5]:

dt = DecisionTreeClassifier(random_state=42)
knn = KNeighborsClassifier(n_neighbors=5)
svm = SVC(probability=True, random_state=42)


## Hard Voting

In [6]:

hard_vote = VotingClassifier(
    estimators=[('dt',dt),('knn',knn),('svm',svm)],
    voting='hard'
)

hard_vote.fit(X_train_scaled,y_train)

pred = hard_vote.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test,pred))


Accuracy: 0.9333333333333333


## Soft Voting

In [7]:

soft_vote = VotingClassifier(
    estimators=[('dt',dt),('knn',knn),('svm',svm)],
    voting='soft'
)

soft_vote.fit(X_train_scaled,y_train)

pred = soft_vote.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test,pred))


Accuracy: 0.9333333333333333


## Stacking Classifier

In [8]:
lr = LogisticRegression()


stack = StackingClassifier(
    estimators=[('dt',dt),('knn',knn),('svm',svm)],
    final_estimator=lr,
    cv=5
)

stack.fit(X_train_scaled,y_train)

pred = stack.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test,pred))


Accuracy: 1.0


## Confusion Matrix

In [9]:

print(confusion_matrix(y_test,pred))


[[10  0  0]
 [ 0 10  0]
 [ 0  0 10]]


## Classification Report

In [10]:

print(classification_report(y_test,pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00        10
           2       1.00      1.00      1.00        10

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



## Predict New Flower

In [11]:

sample = [[5.1,3.5,1.4,0.2]]
sample_scaled = scaler.transform(sample)

prediction = stack.predict(sample_scaled)

print("Predicted Species:", iris.target_names[prediction[0]])


Predicted Species: setosa


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## For more reference

1. 5 Minutes Engineering: https://youtu.be/06aqa-W5YU4
2. AI with Noor: https://youtu.be/5tJq-IsZzTk
3. CampusX **(Must Watch)**: https://youtu.be/O-aDHBGMqXA